# Don Bossing — Free ComfyUI (Kaggle T4 GPU), stable tunnel

Runs ComfyUI + CogVideoX on a **free Kaggle GPU** (T4 15GB, ~30 GPU-hrs/week) and exposes it through a **stable public tunnel** so your local `server.js` can drive it. Setup once, then just re-run when the session expires.

Steps:
0. Run the first code cell (GPU check) and confirm `Tesla T4` appears before the long download.
1. Set this notebook's accelerator to **GPU** (Settings → Accelerator → GPU T4 × 2).
2. (Stable URL) Create a **free ngrok account**, copy your authtoken + pick a subdomain (e.g. `donbossing-video`). In Kaggle: *Add-ons → Secrets* and add `NGROK_AUTHTOKEN` and `NGROK_SUBDOMAIN`. (No-signup alternative: pinggy.io — URL may change each restart.)
3. Run the cells top to bottom.
4. Copy the printed `https://<subdomain>.ngrok-free.dev` URL into `video.config.json` → `comfyUrl` on your PC (set once).
5. On your PC run `npm run local`, open http://localhost:3000, use Section 6.

When the Kaggle session expires, just re-run — the ngrok subdomain is the same, so no config change needed.

**Note:** Do NOT use P100 — PyTorch 2.10 on Kaggle lacks P100 (sm_60) CUDA kernels. T4 (sm_75) works natively.

In [ ]:
# --- 0. Verify GPU + environment (run FIRST, before the long download) ---
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU DETECTED - enable GPU T4 in Settings'
import sys, os
print('Python', sys.version.split()[0])
print('cwd', os.getcwd())


In [ ]:
# --- 1. Install ComfyUI + wrappers ---
import os, subprocess
if not os.path.isdir('ComfyUI'):
    subprocess.run('git clone https://github.com/comfyanonymous/ComfyUI', shell=True)
os.chdir('ComfyUI')
subprocess.run('pip install -q -r requirements.txt', shell=True)
# CogVideoXWrapper requires diffusers>=0.33.1; upgrade so the wrapper can import.
subprocess.run(['pip','install','-q','-U','diffusers>=0.33.1','transformers','accelerate','peft','imageio-ffmpeg'])
def clone_or_pull(url, dest):
    if os.path.isdir(dest):
        print('updating', dest); subprocess.run('git -C ' + dest + ' pull --ff-only', shell=True)
    else:
        subprocess.run('git clone ' + url + ' ' + dest, shell=True)
clone_or_pull('https://github.com/kijai/ComfyUI-CogVideoXWrapper', 'custom_nodes/ComfyUI-CogVideoXWrapper')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite', 'custom_nodes/ComfyUI-VideoHelperSuite')
# Pin the wrapper to the version this workflow was built against.
subprocess.run('git -C custom_nodes/ComfyUI-CogVideoXWrapper checkout fdb8abd2790b5459ddc7066c31861bb0b62e988b', shell=True)
subprocess.run('pip install -q -r custom_nodes/ComfyUI-CogVideoXWrapper/requirements.txt', shell=True)
subprocess.run('pip install -q -r custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt', shell=True)
print('comfyui + wrappers installed')

In [ ]:
# --- 2. Download model weights ---
import os; os.chdir('/kaggle/working/ComfyUI')  # pin cwd so weights land in the right models/ folder
!pip install -q -U huggingface_hub
!mkdir -p models/CogVideo/CogVideoX-5b-I2V models/clip
# Text encoder (t5xxl) for the prompt; CLIPLoader loads it from models/clip.
!hf download comfyanonymous/flux_text_encoders t5xxl_fp8_e4m3fn.safetensors --local-dir models/clip
# CogVideoX transformer + VAE auto-downloaded by DownloadAndLoadCogVideoModel on first generation (~12 GB)
print('t5xxl ready. CogVideoX will auto-download on first generation.')

In [ ]:
# --- 3. Launch ComfyUI in the background + health check ---
import subprocess, time, urllib.request, json, os
os.chdir('/kaggle/working/ComfyUI')

def comfyui_alive():
    try:
        urllib.request.urlopen('http://localhost:8188/system_stats', timeout=5)
        return True
    except Exception:
        return False

# Kill any zombie/defunct python processes from previous runs
os.system('pkill -9 -f "main.py" 2>/dev/null; true')
time.sleep(2)

if comfyui_alive():
    print('ComfyUI already running — skipping launch.')
else:
    log = open('comfy.log','w')
    proc = subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188','--disable-auto-launch'],
                            stdout=log, stderr=subprocess.STDOUT)
    print('ComfyUI launching (pid', proc.pid, ')...')
    time.sleep(30)
    print(open('comfy.log').read()[-2000:])

# Verify it's alive
if comfyui_alive():
    print('ComfyUI OK — ready for tunnel.')
else:
    print('!!! ComfyUI FAILED to start. Check comfy.log above. !!!')

In [ ]:
# --- 4. Stable tunnel: ngrok reserved subdomain (recommended) ---
import os, subprocess, re
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    token = _us.get_secret('NGROK_AUTHTOKEN')
    sub = _us.get_secret('NGROK_SUBDOMAIN')
except Exception:
    token = os.environ.get('NGROK_AUTHTOKEN','')
    sub = os.environ.get('NGROK_SUBDOMAIN','')
if token and sub:
    print('BRANCH: ngrok (stable subdomain)')
    # Install the v3 agent (tgz). v2 (zip) lacks --domain and can't use *.ngrok-free.dev.
    !pkill -9 ngrok 2>/dev/null; true
    !rm -f ngrok ngrok-stable-linux-amd64.zip ngrok3.tgz
    !curl -L -s -o ngrok3.tgz https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
    !tar xzf ngrok3.tgz
    !./ngrok --version
    !./ngrok config add-authtoken {token}
    domain = sub if '.' in sub else (sub + '.ngrok-free.dev')
    tun = subprocess.Popen(['./ngrok','http','--url='+domain,'8188'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = 'https://'+domain
    print('=== STABLE TUNNEL URL (paste once into video.config.json comfyUrl) ===')
    print(url)
    print('=====================================================================')
else:
    # No-signup fallback: pinggy.io (URL may change each restart)
    print('BRANCH: pinggy fallback (NGROK secrets not set)')
    tun = subprocess.Popen(['ssh','-p','443','-o','StrictHostKeyChecking=accept-new','-R0:localhost:8188','a.pinggy.io'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in tun.stdout:
        print(line.rstrip())
        m = re.search(r'(https://[a-z0-9.\-]+\.(?:pinggy\.link|tcp\.ngrok\.io))', line)
        if m:
            print('\n=== TUNNEL URL (paste into video.config.json comfyUrl) ===')
            print(m.group(1))
            print('=========================================================')
            break

In [ ]:
# --- 5. Keep-alive: poll ComfyUI, auto-relaunch if dead ---
import urllib.request, time, json, subprocess, os
os.chdir('/kaggle/working/ComfyUI')
while True:
    try:
        with urllib.request.urlopen('http://localhost:8188/system_stats', timeout=5) as r:
            s = json.load(r)
            print('ComfyUI alive. GPU:', s.get('devices',[{}])[0].get('name','?'))
    except Exception as e:
        print('ComfyUI down:', e, '— relaunching...')
        log = open('comfy.log','a')
        proc = subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188','--disable-auto-launch'],
                                stdout=log, stderr=subprocess.STDOUT)
        print('Relaunched (pid', proc.pid, '), waiting 30s...')
        time.sleep(30)
    time.sleep(60)